# 🚀 Модуль 12. Урок 2. Инструменты и тулы аналитика в LangChain / LlamaIndex

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_12_ai_agents/lesson_02_langchain_llamaindex_for_analytics.ipynb)

## 📌 Бизнес-контекст и методология
Инструмент (Tool) — это Python-функция с описанием входных аргументов, которую модель может вызвать по своему усмотрению.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Описание интерфейса тула аналитика
def calculate_metric_delta(val_current: float, val_previous: float) -> dict:
    """Вычисляет абсолютное и относительное изменение метрики."""
    diff_abs = val_current - val_previous
    diff_rel = (diff_abs / val_previous) if val_previous != 0 else 0.0
    return {"absolute_change": diff_abs, "relative_change_pct": round(diff_rel * 100, 2)}

tool_spec = {
    "name": "calculate_metric_delta",
    "description": "Полезно для расчета дельты между текущим и прошлым периодом",
    "parameters": {
        "val_current": "float",
        "val_previous": "float"
    }
}

res = calculate_metric_delta(120.0, 100.0)
print(f"Результат выполнения тула: {res}")


Результат выполнения тула: {'absolute_change': 20.0, 'relative_change_pct': 20.0}


### 🎯 Практическое задание 1: Проверка расчета дельты
Убедитесь, что относительный прирост составляет 20.0%.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert res["relative_change_pct"] == 20.0
assert res["absolute_change"] == 20.0
print("✅ Тест успешно пройден: тул аналитика протестирован!")


✅ Тест успешно пройден: тул аналитика протестирован!


### 💼 Качество docstring в тулах
Модель решает, вызывать ли инструмент, основываясь исключительно на его docstring. Пишите описания максимально конкретно.